# Download street network

Download street network data from Overture Maps.

In [1]:
from core.generate_streets import read_overture_region_streets

import glob

import geopandas as gpd
import datetime

In [2]:
streets_dir = "/data/uscuni-ulce/overture_streets/"
regions_datadir = "/data/uscuni-ulce/"

In [3]:
country = "extension_2"
regions_v = "4"

## Run in parallel for all non-procesed streets in region_hulls

In [4]:
region_hulls = gpd.read_parquet('/data/uscuni-ulce/' + "regions/" + f"{country}_regions_hull.parquet")

In [5]:
processed_region_ids = [
    int(s.split(".")[0].split("_")[-1]) for s in glob.glob(streets_dir + "*")
]
region_hulls = region_hulls[~region_hulls.index.isin(processed_region_ids)]

In [6]:
region_hulls

,convex_hull
labels,
537200,"POLYGON ((3261700 2299300, 3253900 2306200, 32..."
537429,"POLYGON ((3287200 2336500, 3275700 2337400, 32..."
537600,"POLYGON ((3316700 2228600, 3316100 2228800, 32..."
538559,"POLYGON ((3320600 2242700, 3318200 2243100, 33..."
538918,"POLYGON ((3352600 2285400, 3343400 2290300, 33..."
...,...
849307,"POLYGON ((5322300 4342500, 5320600 4343900, 53..."
850357,"POLYGON ((5364100 4445900, 5320200 4446800, 53..."
856941,"POLYGON ((5328900 3750600, 5302200 3764200, 53..."


In [7]:
def process_and_save(region_hull, region_id):
    print("Processing", region_id, datetime.datetime.now())
    streets = read_overture_region_streets(region_hull, region_id)
    streets.to_parquet(streets_dir + f"streets_{region_id}.pq")

In [ ]:
%%capture cap

from joblib import Parallel, delayed

n_jobs = -1
new = Parallel(n_jobs=n_jobs)(
    delayed(process_and_save)(region_hull.iloc[0], region_id)
    for region_id, region_hull in region_hulls.to_crs(epsg=4326).iterrows()
)

Processing 540231 2026-11-12 18:11:10.587139
Processing 547551 2026-11-12 18:24:08.102235
Processing 545280 2026-11-12 18:11:10.637661
Processing 549793 2026-11-12 18:25:07.331092
Processing 540965 2026-11-12 18:11:10.634285
Processing 550368 2026-11-12 18:25:35.728371
Processing 545900 2026-11-12 18:11:10.623853
Processing 555691 2026-11-12 18:27:45.047941
Processing 538918 2026-11-12 18:11:10.559207
Processing 556455 2026-11-12 18:28:05.543149
Processing 538559 2026-11-12 18:11:10.592541
Processing 552430 2026-11-12 18:26:00.121126
Processing 558896 2026-11-12 18:38:40.279567
Processing 537429 2026-11-12 18:11:10.519217
Processing 549031 2026-11-12 18:24:30.959933
Processing 559120 2026-11-12 18:38:47.640174
Processing 540571 2026-11-12 18:11:10.618775
Processing 548395 2026-11-12 18:24:19.720937
Processing 558687 2026-11-12 18:37:34.188278
Processing 540522 2026-11-12 18:11:10.611750
Processing 554050 2026-11-12 18:26:52.680138
Processing 568913 2026-11-12 18:42:18.993957
Processing

In [9]:
cap.stderr

'/home/martin/dev/urban_taxonomy/.pixi/envs/default/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.\n  warnings.warn(\n'